# RFP vs Proposal Coverage Mapper: Day 2 (file parsing)

**Before running:**
1. **Add Input**: attach ONLY the dataset `rfp-coverage-mapper-final`
2. **Settings > Internet**: On · **Settings > Accelerator**: None
3. Secrets are **not needed today** (Day 2 makes no Gemini calls)

Then click **Run All** (about 1–2 minutes in total).
Every step ends with **✅ STEP X DONE** or **❌ STEP X FAILED: reason**.
If a step fails, it stops the run on purpose (you'll see red text under the ❌ line). Paste the ❌ line to Claude.

## Step 1: Find the project and copy it

In [ ]:
# ===== STEP 1: find the project in the dataset and copy it to the working folder =====
import os, sys, glob, shutil

class StepFailed(Exception):
    pass

def fail(step, reason):
    print(f'❌ STEP {step} FAILED: {reason}')
    raise StepFailed(reason)   # stops Run All on purpose

PROJECT = '/kaggle/working/rfp-coverage-mapper'

# Look for folders that contain check_setup.py AND the official test files
candidates = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/check_setup.py', recursive=True)]
complete = [c for c in candidates if os.path.isfile(os.path.join(c, 'test_data/inputs/proposal_qamar.pptx'))]
print('Folders with check_setup.py:', candidates or 'none')
if not complete:
    fail(1, 'no project folder with test_data/ found. Is the dataset "rfp-coverage-mapper-final" attached (Add Input)?')
SOURCE = sorted(complete, key=lambda c: ('final' not in c, len(c)))[0]   # prefer the "final" dataset
print('Project found at:', SOURCE)

# Fresh copy into the writable working folder (/kaggle/input is read-only)
shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(SOURCE, PROJECT)
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print('Working in:', os.getcwd())

needed = ['test_data/inputs/rfp_qamar.pdf', 'test_data/inputs/rfp_qamar.docx', 'test_data/inputs/proposal_qamar.pptx',
          'test_data/ground_truth/answer_key.json', 'src/llm.py', 'synthetic/scenario.py']
missing = [f for f in needed if not os.path.isfile(f)]
if missing:
    fail(1, f'files missing from the copy: {missing}')
cache_files = len(glob.glob('cache/*.json'))
print(f'Official test files present ✓ | cache entries: {cache_files}')
print('✅ STEP 1 DONE: project copied to', PROJECT)

## Step 2: Install libraries

In [ ]:
# ===== STEP 2: install libraries and check they import =====
import subprocess, importlib
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
                        capture_output=True, text=True)
if result.returncode != 0:
    fail(2, 'pip install failed: ' + result.stderr.strip()[-300:])
print('(pip warnings about dopamine-rl / gradio are harmless and hidden here)')

missing = []
for module in ['pdfplumber', 'docx', 'pptx']:
    try:
        importlib.import_module(module)
    except Exception as e:
        missing.append(f'{module} ({e})')
if missing:
    fail(2, f'libraries not importable: {missing}')
print('✅ STEP 2 DONE: pdfplumber, python-docx and python-pptx are ready')

## Step 3: Save the parser module (`src/parsers.py`) and check it
The first cell below saves the file (you'll see `Writing src/parsers.py`); the second checks it.

In [ ]:
%%writefile src/parsers.py
"""
parsers.py - STAGE 1 of the tool: turn the input files into clean, structured text.

    parse_rfp(path)  -> the RFP (PDF or Word) as a list of "blocks"
    parse_deck(path) -> the proposal deck (PPTX) as a list of slides

A "block" is one heading, paragraph, bullet or table, tagged with the RFP
section it belongs to. Keeping this structure (instead of one big string)
means later stages can say WHERE in the RFP a requirement came from.

No AI is used here: parsing is deterministic, fast, free and repeatable.
"""
import re
from pathlib import Path

import pdfplumber
from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE

# ---------------------------------------------------------------------------
# Safety: the tool must never read the answer key (prevents data leakage)
# ---------------------------------------------------------------------------
BLOCKED_FOLDERS = {"ground_truth"}


def check_allowed(path):
    """Refuse files inside ground_truth/ and files that don't exist."""
    p = Path(path)
    if any(part in BLOCKED_FOLDERS for part in p.parts):
        raise PermissionError(f"Refusing to read '{p}': the tool must never read the answer key (ground_truth).")
    if not p.is_file():
        raise FileNotFoundError(f"File not found: '{p}'")
    return p


# ---------------------------------------------------------------------------
# Text cleaning
# ---------------------------------------------------------------------------
CID = re.compile(r"\(cid:\d+\)")          # PDF symbols pdfplumber can't decode, e.g. bullets -> "(cid:127)"
BULLET_START = re.compile(r"^\s*(\(cid:\d+\)|[•▪◦●\-–*])\s*")
NUMBERED_HEADING = re.compile(r"^\d+(\.\d+)*\.?\s+[A-Z]")   # e.g. "2. Scope of Work"


def clean(text, keep_lines=False):
    """Remove undecodable symbols, non-breaking spaces and repeated spaces.
    keep_lines=True keeps line breaks (used for table rows)."""
    text = CID.sub("", str(text)).replace("\u00a0", " ")
    if keep_lines:
        lines = (re.sub(r"[ \t]+", " ", ln).strip() for ln in text.splitlines())
        return "\n".join(ln for ln in lines if ln)
    return re.sub(r"\s+", " ", text).strip()


def _make_block(blocks, kind, text, section):
    text = clean(text, keep_lines=(kind == "table"))
    if text:
        blocks.append({"id": f"B{len(blocks) + 1:03d}", "type": kind, "section": section, "text": text})


def _finish(blocks, path, fmt):
    """Package blocks into one result, plus a readable full text for the LLM."""
    lines = []
    for b in blocks:
        prefix = {"heading": "## ", "bullet": "• "}.get(b["type"], "")
        lines.append(prefix + b["text"])
    full_text = "\n\n".join(lines)
    return {
        "source": Path(path).name,
        "format": fmt,
        "sections": [b["text"] for b in blocks if b["type"] == "heading"],
        "blocks": blocks,
        "full_text": full_text,
        "word_count": len(full_text.split()),
    }


# ---------------------------------------------------------------------------
# RFP: Word (.docx)
# ---------------------------------------------------------------------------
def _docx_items(doc):
    """Paragraphs AND tables, in the order they appear in the document."""
    for child in doc.element.body.iterchildren():
        tag = child.tag.split("}")[-1]
        if tag == "p":
            yield Paragraph(child, doc)
        elif tag == "tbl":
            yield Table(child, doc)


def parse_docx(path):
    doc = Document(check_allowed(path))
    blocks, section = [], "Front matter"
    for item in _docx_items(doc):
        if isinstance(item, Table):
            rows = [" | ".join(clean(c.text) for c in row.cells) for row in item.rows]
            _make_block(blocks, "table", "\n".join(rows), section)
            continue
        style = item.style.name if item.style is not None else ""
        has_numbering = item._p.pPr is not None and item._p.pPr.numPr is not None
        if style.startswith("Heading"):
            section = clean(item.text) or section
            _make_block(blocks, "heading", item.text, section)
        elif style == "Title":
            _make_block(blocks, "paragraph", item.text, section)
        elif "List" in style or has_numbering:
            _make_block(blocks, "bullet", item.text, section)
        else:
            _make_block(blocks, "paragraph", item.text, section)
    return _finish(blocks, path, "docx")


# ---------------------------------------------------------------------------
# RFP: PDF
# ---------------------------------------------------------------------------
def parse_pdf(path):
    """
    PDFs store text as LINES, not paragraphs, so we rebuild them:
      - a numbered short line ("2. Scope of Work") starts a new section
      - a line starting with a bullet symbol starts a new bullet
      - a short line ending in . : ! ? closes the current paragraph
      - otherwise the line continues the current paragraph
    """
    with pdfplumber.open(check_allowed(path)) as pdf:
        lines = [ln for page in pdf.pages for ln in (page.extract_text() or "").splitlines() if ln.strip()]
    if not lines:
        raise ValueError(f"No text found in '{path}' (is it a scanned PDF?)")

    full_width = max(len(ln) for ln in lines)
    blocks, section = [], "Front matter"
    current, kind = [], "paragraph"

    def flush():
        nonlocal current
        if current:
            _make_block(blocks, kind, " ".join(current), section)
            current = []

    for ln in lines:
        text = ln.strip()
        if NUMBERED_HEADING.match(text) and len(text) < 80 and not text.endswith("."):
            flush()
            section = clean(text)
            _make_block(blocks, "heading", text, section)
            kind = "paragraph"
            continue
        if BULLET_START.match(text):
            flush()
            kind = "bullet"
            text = BULLET_START.sub("", text)
        current.append(text)
        if text.endswith((".", ":", "!", "?")) and len(ln) < 0.85 * full_width:
            flush()
            kind = "paragraph"
    flush()
    return _finish(blocks, path, "pdf")


def parse_rfp(path):
    """Read an RFP from .pdf or .docx (chosen by file extension)."""
    check_allowed(path)          # safety guard ALWAYS runs first, whatever the file type
    suffix = Path(path).suffix.lower()
    if suffix == ".pdf":
        return parse_pdf(path)
    if suffix == ".docx":
        return parse_docx(path)
    raise ValueError(f"Unsupported RFP format '{suffix}'. Use .pdf or .docx (save old .doc files as .docx).")


# ---------------------------------------------------------------------------
# Proposal deck: PowerPoint (.pptx)
# ---------------------------------------------------------------------------
def _walk_shapes(shapes):
    """All shapes in reading order (top to bottom, left to right), opening groups."""
    for shape in sorted(shapes, key=lambda s: ((s.top or 0), (s.left or 0))):
        if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
            yield from _walk_shapes(shape.shapes)
        else:
            yield shape


def parse_deck(path):
    """
    One entry per slide: title, body lines, tables and speaker notes.
    'visible_text' = what the audience sees (title + body + tables).
    Speaker notes are kept separately: the client never sees them, so later
    stages can decide whether they count as "addressing" a requirement.
    """
    check_allowed(path)          # safety guard ALWAYS runs first
    if Path(path).suffix.lower() != ".pptx":
        raise ValueError(f"Unsupported deck format '{Path(path).suffix}'. Use .pptx (save old .ppt files as .pptx).")
    prs = Presentation(path)
    slides = []
    for number, slide in enumerate(prs.slides, start=1):
        title_shape = slide.shapes.title
        title = clean(title_shape.text_frame.text) if title_shape is not None and title_shape.has_text_frame else ""
        body, tables = [], []
        for shape in _walk_shapes(slide.shapes):
            if title_shape is not None and shape.shape_id == title_shape.shape_id:
                continue
            if shape.has_table:
                tables.append([[clean(c.text) for c in row.cells] for row in shape.table.rows])
            elif shape.has_text_frame:
                body.extend(t for t in (clean(p.text) for p in shape.text_frame.paragraphs) if t)
        notes = ""
        if slide.has_notes_slide:
            notes = clean(slide.notes_slide.notes_text_frame.text)

        parts = [f"Title: {title}"] + [f"- {b}" for b in body]
        for table in tables:
            parts += ["Table:"] + [" | ".join(row) for row in table]
        slides.append({
            "slide_number": number,
            "title": title,
            "body": body,
            "tables": tables,
            "notes": notes,
            "visible_text": "\n".join(parts),
        })
    return {"source": Path(path).name, "slide_count": len(slides), "slides": slides}


In [ ]:
# ===== STEP 3: check that parsers.py was saved and loads correctly =====
import importlib
try:
    import src.parsers as parsers
    importlib.reload(parsers)   # always use the version just written
except Exception as e:
    fail(3, f'parsers.py could not be loaded: {e}')
functions = ['parse_rfp', 'parse_pdf', 'parse_docx', 'parse_deck', 'check_allowed', 'clean']
absent = [f for f in functions if not hasattr(parsers, f)]
if absent:
    fail(3, f'functions missing from parsers.py: {absent}')
print('Functions available:', ', '.join(functions))
print('✅ STEP 3 DONE: parsers.py saved and loaded')

## Step 4: Parse the RFP (PDF and Word) and the deck

In [ ]:
# ===== STEP 4: parse the three input files and save the results =====
import json
os.makedirs('output/parsed', exist_ok=True)
results = {}
jobs = [('rfp_from_pdf',  parsers.parse_rfp,  'test_data/inputs/rfp_qamar.pdf'),
        ('rfp_from_docx', parsers.parse_rfp,  'test_data/inputs/rfp_qamar.docx'),
        ('deck',          parsers.parse_deck, 'test_data/inputs/proposal_qamar.pptx')]
for name, parse, path in jobs:
    try:
        results[name] = parse(path)
    except Exception as e:
        fail(4, f'could not parse {path}: {e}')
    with open(f'output/parsed/{name}.json', 'w', encoding='utf-8') as f:
        json.dump(results[name], f, indent=2, ensure_ascii=False)

for name in ('rfp_from_pdf', 'rfp_from_docx'):
    r = results[name]
    print(f"{name}: {r['word_count']} words, {len(r['blocks'])} blocks, {len(r['sections'])} sections")
print(f"deck: {results['deck']['slide_count']} slides")
print('Saved to output/parsed/: rfp_from_pdf.json, rfp_from_docx.json, deck.json')
print('✅ STEP 4 DONE: all three files parsed')

## Step 5: Quality checks

In [ ]:
# ===== STEP 5: quality checks on the parsed output =====
import difflib
pdf, docx_r, deck = results['rfp_from_pdf'], results['rfp_from_docx'], results['deck']
checks = []
def check(ok, label):
    checks.append(ok)
    print(('  ✅ ' if ok else '  ❌ ') + label)

print('RFP')
similarity = difflib.SequenceMatcher(None, pdf['full_text'], docx_r['full_text']).ratio()
check(pdf['sections'] == docx_r['sections'] and len(pdf['sections']) >= 5,
      f"PDF and Word find the same {len(pdf['sections'])} sections")
check(similarity >= 0.95, f'PDF and Word text match ({similarity:.0%} identical)')
check('(cid:' not in pdf['full_text'] + docx_r['full_text'], 'no "(cid:...)" symbols left (PDF bullets cleaned)')
check(all(any(b['type'] == 'bullet' for b in r['blocks']) for r in (pdf, docx_r)), 'bullets recognised in both formats')
check(all(b['section'] for b in pdf['blocks']), 'every block is tagged with its RFP section')

print('Deck')
slides = deck['slides']
check(deck['slide_count'] == 15, f"{deck['slide_count']} slides parsed (expected 15)")
check(all(s['title'] for s in slides), 'every slide has a title')
table_rows = slides[10]['tables'][0] if len(slides) > 10 and slides[10]['tables'] else []
check(len(table_rows) == 5, f'slide 11 table captured ({len(table_rows)} rows incl. header)')
notes = sum(bool(s['notes']) for s in slides)
check(notes >= 1, f'speaker notes captured ({notes}/{len(slides)} slides)')

print('Safety')
try:
    parsers.parse_rfp('test_data/ground_truth/answer_key.json'); refused = False
except PermissionError:
    refused = True
check(refused, 'the tool REFUSES to read the answer key (ground_truth)')
try:
    parsers.parse_rfp('requirements.txt'); rejected = False
except ValueError:
    rejected = True
check(rejected, 'unsupported file types are rejected with a clear message')

failed = checks.count(False)
if failed:
    fail(5, f'{failed} quality check(s) failed (see ❌ above)')
print(f'✅ STEP 5 DONE: all {len(checks)} quality checks passed')

## Step 6: Preview what the tool sees

In [ ]:
# ===== STEP 6: preview: what the tool now "sees" =====
import pdfplumber
with pdfplumber.open('test_data/inputs/rfp_qamar.pdf') as p:
    raw_lines = [l for page in p.pages for l in (page.extract_text() or '').splitlines()]
raw_bullet = next((l for l in raw_lines if '(cid:' in l), '(none)')
print('BEFORE cleaning (raw PDF line):', raw_bullet)
first_bullet = next(b for b in pdf['blocks'] if b['type'] == 'bullet')
print('AFTER cleaning (parsed block): ', first_bullet['id'], first_bullet['type'], '|', first_bullet['text'][:90])

print('\n--- RFP blocks in section 2 (from the PDF) ---')
for b in pdf['blocks']:
    if b['section'].startswith('2.'):
        print(f"{b['id']} [{b['type']}] {b['text'][:110]}{'...' if len(b['text']) > 110 else ''}")

print('\n--- Slide 11 as the tool sees it ---')
print(slides[10]['visible_text'])
print('Speaker notes:', slides[10]['notes'])
print('\n✅ STEP 6 DONE: preview shown')

## Step 7: Download

In [ ]:
# ===== STEP 7: save everything as a zip to download =====
zip_path = shutil.make_archive('/kaggle/working/rfp_coverage_mapper_day2', 'zip', '/kaggle/working', 'rfp-coverage-mapper')
size_kb = os.path.getsize(zip_path) // 1024
if size_kb == 0:
    fail(7, 'zip file is empty')
print(f'Saved {zip_path} ({size_kb} KB): code + test data + cache + output/parsed/')
print('✅ STEP 7 DONE: download rfp_coverage_mapper_day2.zip from the Output panel')